<a href="https://colab.research.google.com/github/epmbanten/Tugas-AI-602225073-Dody-Suhendra/blob/main/DodySuhendra_6022251073_ImageProc_Basic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import data
from skimage.measure import block_reduce

# ==========================================
# PERSIAPAN DATA
# Menggunakan gambar sampel bawaan (Astronaut)
# ==========================================
image_rgb = data.astronaut()
image_gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)

# Fungsi bantuan untuk plot gambar
def plot_image(img, title, cmap=None):
    plt.imshow(img, cmap=cmap)
    plt.title(title)
    plt.axis('off')

plt.figure(figsize=(15, 20))

# ==========================================
# 1. MEMISAHKAN KOMPONEN R, G, B
# Memahami bahwa gambar berwarna adalah tensor dengan depth/channel = 3
# ==========================================
R, G, B = cv2.split(image_rgb)

# Membuat gambar kosong untuk visualisasi warna aslinya
zeros = np.zeros(image_rgb.shape[:2], dtype="uint8")
img_R = cv2.merge([R, zeros, zeros])
img_G = cv2.merge([zeros, G, zeros])
img_B = cv2.merge([zeros, zeros, B])

plt.subplot(6, 3, 1); plot_image(image_rgb, "Original RGB")
plt.subplot(6, 3, 2); plot_image(img_R, "Red Channel")
plt.subplot(6, 3, 3); plot_image(img_G, "Green Channel")

# ==========================================
# 2. MENGATUR BRIGHTNESS & CONTRAST
# Rumus: Output = alpha * Input + beta
# alpha (>0) untuk Contrast, beta untuk Brightness
# ==========================================
alpha = 1.5 # Contrast control (1.0-3.0)
beta = 40   # Brightness control (0-100)
adjusted_img = cv2.convertScaleAbs(image_rgb, alpha=alpha, beta=beta)

plt.subplot(6, 3, 4); plot_image(image_gray, "Original Grayscale", cmap='gray')
plt.subplot(6, 3, 5); plot_image(adjusted_img, f"Brightness & Contrast\n(alpha={alpha}, beta={beta})")
plt.subplot(6, 3, 6); plot_image(img_B, "Blue Channel")

# ==========================================
# 3 & 4. KONVOLUSI: BLURRING & SHARPENING
# Ini adalah inti dari Convolutional Layer pada CNN
# ==========================================
# Kernel Blurring (Rata-rata / Average Filter)
kernel_blur = np.ones((5, 5), np.float32) / 25
img_blur = cv2.filter2D(image_rgb, -1, kernel_blur)

# Kernel Sharpening
kernel_sharpen = np.array([
    [0, -1,  0],
    [-1, 5, -1],
    [0, -1,  0]
])
img_sharpen = cv2.filter2D(image_rgb, -1, kernel_sharpen)

plt.subplot(6, 3, 7); plot_image(image_rgb, "Original (Pre-Convolution)")
plt.subplot(6, 3, 8); plot_image(img_blur, "Convolution: Blurring")
plt.subplot(6, 3, 9); plot_image(img_sharpen, "Convolution: Sharpening")

# ==========================================
# 5. EDGE & CORNER DETECTION
# Fitur awal yang biasanya dideteksi oleh layer CNN pertama
# ==========================================
# Sobel Edge Detection (Deteksi garis horizontal dan vertikal)
sobel_x = cv2.Sobel(image_gray, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(image_gray, cv2.CV_64F, 0, 1, ksize=3)
sobel_combined = cv2.magnitude(sobel_x, sobel_y)

# Canny Edge Detection
edges_canny = cv2.Canny(image_gray, 100, 200)

# Harris Corner Detection
gray_float = np.float32(image_gray)
corners = cv2.cornerHarris(gray_float, 2, 3, 0.04)
# Dilatasi untuk memperjelas titik sudut
corners_dilated = cv2.dilate(corners, None)
img_corners = image_rgb.copy()
img_corners[corners_dilated > 0.01 * corners_dilated.max()] = [255, 0, 0] # Tandai sudut dgn warna merah

plt.subplot(6, 3, 10); plot_image(sobel_combined, "Sobel Edges", cmap='gray')
plt.subplot(6, 3, 11); plot_image(edges_canny, "Canny Edges", cmap='gray')
plt.subplot(6, 3, 12); plot_image(img_corners, "Harris Corners (Red)")

# ==========================================
# 6. DASAR LAINNYA UNTUK CNN: MAX POOLING & THRESHOLDING (ReLU)
# ==========================================
# Thresholding (Menganalogikan fungsi aktivasi ReLU yang membuang nilai negatif/rendah)
_, img_thresh = cv2.threshold(image_gray, 127, 255, cv2.THRESH_BINARY)

# Max Pooling (Mengurangi dimensi spatial, umum di CNN)
# Mengambil nilai maksimal dari setiap blok 4x4
img_max_pool = block_reduce(image_gray, block_size=(4, 4), func=np.max)

plt.subplot(6, 3, 13); plot_image(image_gray, "Original Grayscale", cmap='gray')
plt.subplot(6, 3, 14); plot_image(img_thresh, "Thresholding (Biner)", cmap='gray')
plt.subplot(6, 3, 15); plot_image(img_max_pool, "Max Pooling (4x4)", cmap='gray')

plt.tight_layout()
plt.show()

# Mengecek ukuran (dimensi) gambar
print("Dimensi gambar aslinya (RGB):", image_rgb.shape)
print("Dimensi gambar grayscale:", image_gray.shape)